## Notebook to Analyse the Agentic Pipeline Output and Identify Improvements

## Round 1: 23/5/2026 5:30 PM

In [1]:
import sys
import os
import pandas as pd
import json
from IPython.display import display, HTML
from dotenv import load_dotenv
load_dotenv()
%load_ext autoreload
%autoreload 2
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler


/home/rishabhs/miniconda3/envs/langfuse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
langfuse = Langfuse(
  secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
  public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
  host=os.getenv('LANGFUSE_HOST')
)

In [3]:
# Need source agents to re-run for error records and log in LangFuse
sys.path.append(os.path.abspath('..'))
from src.agents import run_jd_extraction_agent, run_resume_extraction_agent, run_evaluator_agent

### Find Top False Positive and False Negative Records

In [4]:
def print_analysis_report(title, dataframe):
    """Formats the edge-case records for easy reading in the notebook."""
    print(f"\n{'='*60}")
    print(f"{title.upper()}")
    print(f"{'='*60}\n")
    
    if dataframe.empty:
        print("No records found for this category.")
        return
        
    for idx, row in dataframe.iterrows():
        print(f"CV ID: {row['CV_ID']} | Role: {row['Target_Role']}")
        print(f"Ground Truth: {row['Ground_Truth_Tier']} --> LLM Score: {row['Match_Score']} ({row['Recommendation']})")
        print(f"\nLLM Justification:\n{row['Justification']}")
        print(f"\nIdentified Strengths:\n{row['Strengths']}")
        print(f"\nCritical Gaps:\n{row['Critical_Gaps']}")
        print("-" * 60 + "\n")

In [4]:
# Load the Evaluation Data
file_path = '../datasets/final_evaluation_report.xlsx'
df = pd.read_excel(file_path, sheet_name='Candidate_Evaluations')


TOP_TIER_LABEL = 'Strong Match'  
BOTTOM_TIER_LABEL = 'Bad Match'  

print(f"Total records loaded: {len(df)}")

# ==========================================
# FIND TOP 5 FALSE POSITIVES
# (Ground Truth is Bad, but LLM gave a High Score)
# ==========================================
false_positives = df[df['Ground_Truth_Tier'] == BOTTOM_TIER_LABEL].sort_values(by='Match_Score', ascending=False).head(5)

# ==========================================
# FIND TOP 5 FALSE NEGATIVES
# (Ground Truth is Strong, but LLM gave a Low Score)
# ==========================================
false_negatives = df[df['Ground_Truth_Tier'] == TOP_TIER_LABEL].sort_values(by='Match_Score', ascending=True).head(5)



# Display the reports
print_analysis_report("Top 5 False Positives (Over-Scored Candidates)", false_positives)
print_analysis_report("Top 5 False Negatives (Under-Scored Candidates)", false_negatives)

Total records loaded: 60

TOP 5 FALSE POSITIVES (OVER-SCORED CANDIDATES)

No records found for this category.

TOP 5 FALSE NEGATIVES (UNDER-SCORED CANDIDATES)

CV ID: 6 | Role: Senior C# Developer
Ground Truth: Strong Match --> LLM Score: 40 (Borderline)

LLM Justification:
The candidate has some relevant skills, but falls short in several key areas. They have experience with C#, which is a requirement, but lack the necessary experience and skills in other areas such as MVC Architecture, API development, ERP development, OOPS and Design Principles, RestAPI Integration, SQL Server, vb .net, and Azure DevOps.

Identified Strengths:
- C# is listed in the candidate's technical skills

Critical Gaps:
- MVC Architecture
- API development
- ERP development
- OOPS and Design Principles
- RestAPI Integration
- SQL Server
- vb .net
- Azure DevOps
------------------------------------------------------------

CV ID: 16 | Role: AI Research Scientist
Ground Truth: Strong Match --> LLM Score: 60 (Pro

In [5]:
false_negatives.head()

,Target_Role,Rank_in_Search,CV_ID,Ground_Truth_Tier,Recommendation,Match_Score,Justification,Strengths,Critical_Gaps
38,Senior C# Developer,9,6,Strong Match,Borderline,40,"The candidate has some relevant skills, but fa...",- C# is listed in the candidate's technical sk...,- MVC Architecture\n- API development\n- ERP d...
0,AI Research Scientist,1,16,Strong Match,Proceed to Interview,60,The candidate has some relevant technical skil...,"- Expertise in Python, a must-have skill for t...",- Lack of experience in large scale distribute...
5,AI Research Scientist,6,5,Strong Match,Borderline,60,The candidate has some relevant skills and exp...,"- Proficient in Python, which is a must-have s...",- Lack of explicit mention of AI research or s...
4,AI Research Scientist,5,6,Strong Match,Proceed to Interview,60,The candidate has some relevant technical skil...,"- Proficient in Python, which is a must-have s...",- Lack of explicit mention of AI research as a...
22,Marketing Data Analyst,3,6,Strong Match,Proceed to Interview,60,The candidate has a strong background in SQL a...,- Strong SQL skills with a demonstrated backgr...,"- Lack of experience with Confluence, Jira, Lu..."


### Re-run Evaliation for False Positive and Log in LangFuse

In [5]:
# Load the original raw text dataset
raw_data_path = '../datasets/jd_resume_pairs.csv'
df_raw = pd.read_csv(raw_data_path)

In [8]:
TARGET_CV_ID = 16
# Extract source data for False Negative record
raw_jd = df_raw.loc[TARGET_CV_ID, 'source_jd_description']
raw_cv = df_raw.loc[TARGET_CV_ID, 'generated_resume_text']
target_role = df_raw.loc[TARGET_CV_ID, 'source_jd_title']
actual_tier = df_raw.loc[TARGET_CV_ID, 'assigned_tier']
print(f"--- ISOLATING CV_ID: {TARGET_CV_ID} ({target_role}) ---")
# LangFuse Setup for Debugging
debug_handler = CallbackHandler()

debug_config = {
    "callbacks": [debug_handler],
    "run_name": f"DEBUG_RUN_CV_{TARGET_CV_ID}",
    "tags": ["Deep_Dive", "False_Negative", f"CV_{TARGET_CV_ID}"],
    "metadata": {
        
        "langfuse_session_id": "Manual_Error_Analysis_v3",
        "investigation_reason": "Ground truth was Strong Match, but LLM failed it."
    }
}

--- ISOLATING CV_ID: 16 (AI Research Scientist) ---


In [9]:
## Running the agents
print("Running JD Extractor...")
jd_json = run_jd_extraction_agent(raw_jd, config=debug_config)

print("Running CV Extractor...")
cv_json = run_resume_extraction_agent(raw_cv, config=debug_config)

print("Running Evaluator...")
eval_result = run_evaluator_agent(jd_json, cv_json, config=debug_config)

# Print Results
print("\n=== DEBUG EXECUTION COMPLETE ===")
print(f"Ground Truth: {actual_tier}")
print(f"LLM Recommendation: {eval_result.get('recommendation')}")
print(f"LLM Score: {eval_result.get('match_score')}")




Running JD Extractor...
Running CV Extractor...
Running Evaluator...

=== DEBUG EXECUTION COMPLETE ===
Ground Truth: Strong Match
LLM Recommendation: Proceed to Interview
LLM Score: 70


### The Evaliator identified the following as gaps
- "Lack of experience in large scale distributed systems, a must-have requirement."
-  "No mention of PyTorch, Transformers, or Deepspeed, all of which are must-have skills for the role."
### A read of the CV reveals
- skills like TensorFlow, JAX and Transformer Architecture variants are mentioned

## Fix Added: Changed the Evaluator Prompt to consider Semantic Equivalence and not look for keywords only

### With this change, re-running the scoring pipeline produced the following summary result
=== PIPELINE PERFORMANCE SUMMARY ===
Spearman Rank Correlation (0 to 1): 0.403

Score Distribution by Tier:
| Ground_Truth_Tier   |   Average_Match_Score |   Median_Match_Score |   Sample_Size |
|:--------------------|----------------------:|---------------------:|--------------:|
| Average Match       |               78.2812 |                   85 |            32 |
| Strong Match        |               82.3214 |                   85 |            28 |

### The Scoring prompt is not working correctly and needs to be fine tuned

## Fix Added: Updated the Prompt to use Semantic Equivalence but defined scoring buckets and parameters 

### With this change, re-running the pipeline produced the following result which was worse than the previous result

=== PIPELINE PERFORMANCE SUMMARY ===
Spearman Rank Correlation (0 to 1): 0.343

Score Distribution by Tier:
| Ground_Truth_Tier   |   Average_Match_Score |   Median_Match_Score |   Sample_Size |
|:--------------------|----------------------:|---------------------:|--------------:|
| Average Match       |               79.6562 |                   84 |            32 |
| Strong Match        |               84.6786 |                   85 |            28 |


## Fix Added: Changed the order of fields in the Pydantic class for Evaluation Result
### Moved the Match_score to the bottom of the class instead of top of the class so that the model matches skills first, prepares justification and then assigns score

## With this change, re-running the pipeline produced the following result which was marginally better than the previous results
=== PIPELINE PERFORMANCE SUMMARY ===
Spearman Rank Correlation (0 to 1): 0.402

Score Distribution by Tier:
| Ground_Truth_Tier   |   Average_Match_Score |   Median_Match_Score |   Sample_Size |
|:--------------------|----------------------:|---------------------:|--------------:|
| Average Match       |               64.2812 |                   69 |            32 |
| Strong Match        |               72.8929 |                   70 |            28 |

### Observations
 - The median score comes down due to stricter evaluation criteria
 - Chain of thought implementation to match every skill before assigning the score is working well. 
 - But as a result the evaluator agent is penalising CVs with even 1 missing skill.
 - Future improvement opportunities: 
    - Use a bigger model for Evaluation
    - Use Human in the loop to ensure every JD lists only 3-4 must have skills

## Round 2: 23/5/2026 8:30 PM

In [10]:
# Load the Evaluation Data
file_path = '../datasets/final_evaluation_report.xlsx'
df = pd.read_excel(file_path, sheet_name='Candidate_Evaluations')


TOP_TIER_LABEL = 'Strong Match'  
BOTTOM_TIER_LABEL = 'Bad Match'  

print(f"Total records loaded: {len(df)}")

# ==========================================
# FIND TOP 5 FALSE POSITIVES
# (Ground Truth is Bad, but LLM gave a High Score)
# ==========================================
false_positives = df[df['Ground_Truth_Tier'] == BOTTOM_TIER_LABEL].sort_values(by='Match_Score', ascending=False).head(5)

# ==========================================
# FIND TOP 5 FALSE NEGATIVES
# (Ground Truth is Strong, but LLM gave a Low Score)
# ==========================================
false_negatives = df[df['Ground_Truth_Tier'] == TOP_TIER_LABEL].sort_values(by='Match_Score', ascending=True).head(5)



# Display the reports
print_analysis_report("Top 5 False Positives (Over-Scored Candidates)", false_positives)
print_analysis_report("Top 5 False Negatives (Under-Scored Candidates)", false_negatives)

Total records loaded: 60

TOP 5 FALSE POSITIVES (OVER-SCORED CANDIDATES)

No records found for this category.

TOP 5 FALSE NEGATIVES (UNDER-SCORED CANDIDATES)

CV ID: 5 | Role: AI Research Scientist
Ground Truth: Strong Match --> LLM Score: 50 (Borderline)

LLM Justification:
The candidate has a strong foundation in Python and some familiarity with deep learning frameworks, but lacks critical skills in AI research, large-scale distributed systems, and specific deep learning frameworks.

Identified Strengths:
- Proficient in Python programming language.
- Familiar with deep learning frameworks such as PyTorch (equivalent to pandas, NumPy, and scikit-learn).

Critical Gaps:
- Lack of experience in AI research
- No experience in large scale distributed systems
- Missing skills in C/C++/CUDA programming languages
- No knowledge of Transformers or Deepspeed deep learning frameworks
------------------------------------------------------------

CV ID: 6 | Role: AI Research Scientist
Ground Tr

## Now the top 5 False positives are all strong candidates for a ***different job category***. The Evaluator agent correctly identified them as low matches 

## Fix Added: For all CVs from a role that is not the profile being matched against, set the ground label to Bad Match
- E.g. if a CV that is a "Strong Match" for Sr. Data Scientist is retrieved for the JD of an AI Research Engineer, change the ground truth to Bad Match
- Now the Evaluation Results are as follows:

=== PIPELINE PERFORMANCE SUMMARY ===
Spearman Rank Correlation (0 to 1): 0.493

Score Distribution by Tier:
| Ground_Truth_Tier   |   Average_Match_Score |   Median_Match_Score |   Sample_Size |
|:--------------------|----------------------:|---------------------:|--------------:|
| Average Match       |               68.25   |                   70 |            12 |
| Bad Match           |               63.9167 |                   70 |            36 |
| Strong Match        |               81.5    |                   83 |            12 |


### Final Conclusion and Summary
- The Spearman Rank Correlation improved from 0.3 to 0.5 through the following changes
    - Changed the Evaluator Prompt to consider semantic variations as matches
    - Changed the Evaluation Result Object to implement Chain of Thought (by forcing the evaluator to match skills, prepare justification and then assign match score )
    - Corrected Ground truth labels to "Bad match" if CVs were fetched from a different role category
- The changes have also resulted in better separation of Strong Match CVs from others
    - Further tweaking and generalisation can help build a ***cut-off** match score to automatically select strong candidates